In [ ]:
# 🔷 Question 2: Design a Multi-Agent Workflow with LangGraph (25 Marks)
# 🧩 Scenario
# You are building an AI-powered customer support system for a fintech company.

# The system must handle:

# Transaction queries
# Fraud detection flags
# Refund requests
# General FAQs
# The company wants:

# High accuracy
# Step-by-step reasoning
# Ability to retry if answer is incorrect
# Modular, scalable architecture
# 💻 Task
# Design and implement a multi-agent workflow using LangGraph (or similar framework).

# ✅ 1. Agent Design
# Define at least 3 agents, such as:

# Retrieval Agent
# Reasoning/Answer Agent
# Validation Agent
# Explain briefly (in comments or code):

# Each agent’s role
# Input/output
# ✅ 2. Graph Workflow Implementation
# Write code or pseudo-code to:

# Define state
# Add nodes (agents)
# Define edges
# Implement conditional logic
# 👉 Must include:

# Retry loop if validation fails
# Clear start and end states
# ✅ 3. State Management
# Show how state evolves across steps:

# Query
# Context
# Intermediate reasoning
# Final answer
# Validation flag
# ✅ 4. Task Delegation & Communication
# Demonstrate:

# How agents pass information
# How decisions are made between agents
# 🎯 Expected Outcome
# A clear multi-step, graph-based agent system that:

# Handles complex queries
# Demonstrates reasoning + validation
# Uses proper orchestration 

In [3]:
import os
from typing import TypedDict
from dotenv import load_dotenv

from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq

# LOAD API KEY

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key=api_key,
    model="llama-3.1-8b-instant"
)

# STATE DEFINITION

class SupportState(TypedDict):
    query: str
    context: str
    reasoning: str
    answer: str
    validated: bool
    retry_count: int


# RETRIEVAL AGENT

def retrieval_agent(state: SupportState):

    query = state["query"]

    faq_db = {
        "refund": "Refunds are processed within 5-7 business days.",
        "transaction declined": "Transactions can fail due to insufficient balance or bank restrictions.",
        "fraud": "Suspicious transactions are flagged by automated fraud detection systems."
    }

    context = "No relevant policy found."

    for key in faq_db:
        if key in query.lower():
            context = faq_db[key]

    state["context"] = context

    print("\n[Retrieval Agent]")
    print("Context Retrieved:", context)

    return state


# REASONING AGENT

def reasoning_agent(state: SupportState):

    query = state["query"]
    context = state["context"]

    prompt = f"""
You are a fintech customer support assistant.

User Question:
{query}

Context:
{context}

Explain reasoning briefly and provide the final answer.

Format:
Reasoning:
Answer:
"""

    response = llm.invoke(prompt)

    output = response.content

    reasoning = ""
    answer = ""

    if "Answer:" in output:
        parts = output.split("Answer:")
        reasoning = parts[0]
        answer = parts[1]
    else:
        answer = output

    state["reasoning"] = reasoning.strip()
    state["answer"] = answer.strip()

    print("\n[Reasoning Agent]")
    print("Reasoning:", state["reasoning"])
    print("Answer:", state["answer"])

    return state


# VALIDATION AGENT

def validation_agent(state: SupportState):

    answer = state["answer"]
    context = state["context"]

    prompt = f"""
Check if the answer is consistent with the context.

Context:
{context}

Answer:
{answer}

Respond only with:
VALID
or
INVALID
"""

    response = llm.invoke(prompt)

    result = response.content.strip()

    state["validated"] = "VALID" in result.upper()

    print("\n[Validation Agent]")
    print("Validation Result:", state["validated"])

    return state


# ROUTER (RETRY LOGIC)

def validation_router(state: SupportState):

    if state["validated"]:
        return "end"

    if state["retry_count"] >= 2:
        print("\nMax retries reached.")
        return "end"

    state["retry_count"] += 1

    print("\nRetrying reasoning... Attempt:", state["retry_count"])

    return "reasoning"


# BUILD LANGGRAPH WORKFLOW

graph = StateGraph(SupportState)

graph.add_node("retrieval", retrieval_agent)
graph.add_node("reasoning", reasoning_agent)
graph.add_node("validation", validation_agent)

graph.set_entry_point("retrieval")

graph.add_edge("retrieval", "reasoning")
graph.add_edge("reasoning", "validation")

graph.add_conditional_edges(
    "validation",
    validation_router,
    {
        "reasoning": "reasoning",
        "end": END
    }
)

app = graph.compile()


# RUN SYSTEM

initial_state = {
    "query": "Why was my transaction declined?",
    "context": "",
    "reasoning": "",
    "answer": "",
    "validated": False,
    "retry_count": 0
}

result = app.invoke(initial_state)

print("FINAL ANSWER")


print(result["answer"])


[Retrieval Agent]
Context Retrieved: Transactions can fail due to insufficient balance or bank restrictions.

[Reasoning Agent]
Reasoning: Reasoning:
There could be two main reasons why your transaction was declined: 
1. Insufficient balance: Your account balance might be lower than the transaction amount, resulting in a decline.
2. Bank restrictions: Your bank might have placed a hold on your account or restricted certain types of transactions, causing the decline.
Answer: To confirm the reason for the declined transaction, I would need to check the details of the transaction. However, as a general suggestion, please check your account balance and contact your bank to see if there are any restrictions on your account.

[Validation Agent]
Validation Result: True
FINAL ANSWER
To confirm the reason for the declined transaction, I would need to check the details of the transaction. However, as a general suggestion, please check your account balance and contact your bank to see if there a